In [0]:
import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:
import os

mlruns_path = "/Volumes/workspace/ecommerce/ecommerce_data/mlruns"
os.makedirs(mlruns_path, exist_ok=True)

print("Using mlruns_path:", mlruns_path)


In [0]:
import mlflow

mlflow.set_tracking_uri(f"file:{mlruns_path}")
mlflow.set_experiment("day12_mlflow")

with mlflow.start_run(run_name="sanity"):
    mlflow.log_param("p", 1)
    mlflow.log_metric("m", 0.5)

print("tracking_uri:", mlflow.get_tracking_uri())

In [0]:
# Build a training dataset (product-day grain)
events = spark.table("silver.events").withColumn("event_date", F.to_date("event_ts"))

prod_day = (
    events.groupBy("event_date", "product_id", "category_code", "brand")
    .agg(
        F.sum(F.when(F.col("event_type")=="view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type")=="cart", 1).otherwise(0)).alias("carts"),
        F.sum(F.when(F.col("event_type")=="purchase", 1).otherwise(0)).alias("purchases"),
        F.sum(F.when(F.col("event_type")=="purchase", F.col("price")).otherwise(0)).alias("revenue"),
        F.avg(F.when(F.col("event_type")=="purchase", F.col("price"))).alias("avg_purchase_price"),
    )
    .fillna({"avg_purchase_price": 0.0})
    .withColumn("conversion_rate", F.when(F.col("views")==0, F.lit(0.0)).otherwise(F.col("purchases")/F.col("views")))
)

prod_day.display()

In [0]:
# Persisting as Gold so it’s reproducible:
(prod_day.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("gold.ml_training_product_day"))


In [0]:
# Train/test split (time-based, not random)
df = spark.table("gold.ml_training_product_day")

max_date = df.agg(F.max("event_date")).first()[0]
cutoff = df.select(F.date_sub(F.lit(max_date), 7).alias("cutoff")).first()["cutoff"]  # last 7 days as test

train_df = df.filter(F.col("event_date") < F.lit(cutoff))
test_df  = df.filter(F.col("event_date") >= F.lit(cutoff))

print("train rows:", train_df.count())
print("test rows:", test_df.count())
print("cutoff:", cutoff)

In [0]:
# Assemble features
## - Keeping feature set minimal and explainable.
feature_cols = ["views", "carts", "revenue", "avg_purchase_price", "conversion_rate"]
label_col = "purchases"

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

train_vec = assembler.transform(train_df).select("features", F.col(label_col).alias("label"))
test_vec  = assembler.transform(test_df).select("features", F.col(label_col).alias("label"))


In [0]:
# Baseline model: Linear Regression + MLflow tracking
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

with mlflow.start_run(run_name="lr_baseline_product_day"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("label", label_col)
    mlflow.log_param("features", ",".join(feature_cols))
    mlflow.log_param("split_strategy", "time_based")
    mlflow.log_param("test_window_days", 7)

    lr = LinearRegression(featuresCol="features", labelCol="label")
    lr_model = lr.fit(train_vec)

    preds = lr_model.transform(test_vec)
    rmse = evaluator_rmse.evaluate(preds)
    r2 = evaluator_r2.evaluate(preds)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    # Log Spark model
    mlflow.spark.log_model(lr_model, artifact_path="model", dfs_tmpdir="/Volumes/workspace/ecommerce/ecommerce_data/mlruns/tmp", pip_requirements=["pyspark==4.0.0", "mlflow"])

print(f"LR rmse={rmse:.4f}, r2={r2:.4f}")

In [0]:
# Better baseline: Random Forest + MLflow run comparison
with mlflow.start_run(run_name="rf_v1_product_day"):
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("label", label_col)
    mlflow.log_param("features", ",".join(feature_cols))
    mlflow.log_param("split_strategy", "time_based")
    mlflow.log_param("test_window_days", 7)

    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="label",
        numTrees=100,
        maxDepth=8,
        seed=42
    )
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 8)

    rf_model = rf.fit(train_vec)

    preds = rf_model.transform(test_vec)
    rmse = evaluator_rmse.evaluate(preds)
    r2 = evaluator_r2.evaluate(preds)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.spark.log_model(lr_model, artifact_path="model", dfs_tmpdir="/Volumes/workspace/ecommerce/ecommerce_data/mlruns/tmp", pip_requirements=["pyspark==4.0.0", "mlflow"])

print(f"RF rmse={rmse:.4f}, r2={r2:.4f}")